In [ ]:
# 以Mg + Gd + Y + Zn + Mn
# Mg 基体固定在 80–97%；

# Gd + Y + Zn总和 20%  Mn小于1% 0.2-1
# 以Mg + Gd + Y + Zn + Mn为起点
# 产生对应特征，包括24种元素和对应的  hall_petch_relation(3个), grain_size, habit_plane(1-7)  

# hall-petch relation

# Grain_Size
# 晶粒尺寸

# wenalloy: 
# Interant d electrons

# habit plane
# 1-7

# magpie 
# MagpieData avg_dev MeltingT

# meredig 
# mean AtomicRadius




In [2]:
import pandas as pd
import numpy as np
from pymatgen.core import Composition
from matminer.featurizers.composition.element import ElementFraction

# -------------------------------
# 元素集合 & Grain Size 设置
# -------------------------------
elements = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
            'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
grain_sizes = list(range(30, 9, -3))  # [30, 27, ..., 12]

import numpy as np
import pandas as pd

grain_sizes = [12]

data = []

for mn in [0.2, 0.4, 0.6, 0.8, 1.0]:
    remaining = 100.0 - mn
    for zn in np.arange(0.0, min(4.0, remaining) + 0.001, 0.5):
        for y in np.arange(0.0, remaining - zn + 0.001, 0.5):
            for gd in np.arange(0.0, remaining - zn - y + 0.001, 0.5):
                mg = remaining - zn - y - gd
                if mg <= 80:
                    continue
                for grain in grain_sizes:
                    comp = {el: 0.0 for el in elements}
                    comp['Mg'] = round(mg, 2)
                    comp['Gd'] = round(gd, 2)
                    comp['Y'] = round(y, 2)
                    comp['Zn'] = round(zn, 2)
                    comp['Mn'] = round(mn, 2)
                    comp['Grain Size'] = grain
                    data.append(comp)

# 统计数量并生成 DataFrame
df_wt = pd.DataFrame(data)
print(f"✅ 共生成 {len(df_wt)} 条样本")


# -------------------------------
# 构建 Label（追踪编号，不包含 Grain Size）
# -------------------------------
def build_label(row):
    return ''.join(f"{el}-{row[el]}" for el in elements if row[el] > 0 and el != 'Grain Size')

df_wt['Label'] = df_wt.apply(build_label, axis=1)

# -------------------------------
# 转为 Composition（用于 molar fraction）
# -------------------------------
def to_comp(x):
    return Composition.from_weight_dict({el: x[el] for el in elements})

df_wt['composition'] = df_wt.apply(to_comp, axis=1)

# -------------------------------
# 计算摩尔比例（ElementFraction）
# -------------------------------
df_frac = ElementFraction().featurize_dataframe(df_wt[['composition']].copy(), col_id='composition')
df_frac = df_frac.drop(columns='composition')  # 删除中间列

# -------------------------------
# 合并：Label + Grain Size + 摩尔比例
# -------------------------------
df_final = pd.concat([df_wt[['Label', 'Grain Size']], df_frac], axis=1)

# -------------------------------
# 保存结果
# -------------------------------
output_path = '1composition_grid_with_fraction_and_grainsize.xlsx'
df_final.to_excel(output_path, index=False)

print(f"✅ 共生成 {len(df_final)} 条样本，已保存为：{output_path}")


✅ 共生成 28833 条样本


ElementFraction:   0%|          | 0/28833 [00:00<?, ?it/s]

✅ 共生成 28833 条样本，已保存为：1composition_grid_with_fraction_and_grainsize.xlsx


In [ ]:
# hall-petch筛选
import pandas as pd
from strength_calculation import calculate_Strength
elements = pd.read_excel('1composition_grid_with_fraction_and_grainsize.xlsx', header = 0, sheet_name = 'Sheet1' )
# featurizer = ElementProperty.from_preset("magpie")

# from matminer.featurizers.composition.composite import ElementProperty
# 选出小于16的
df_strength = calculate_Strength(elements[elements['Grain Size']<16])
# 筛选出强度 > 225 的样本
high_strength_df = df_strength[df_strength["Calculated Yield Strength"] > 225]

# 打印筛选后的样本
print(high_strength_df)
# print(df_strength)

KeyError: 'Grain sizes'

In [7]:
# 特定元素体系的与晶粒尺寸呈最直接的关系
high_strength_df.to_excel('2hall_petch_relation_filter.xlsx', index = False) 
print(high_strength_df.shape[0])

62


In [1]:
import pandas as pd
import numpy as np
from pymatgen.core import Composition
from matminer.featurizers.composition.element import ElementFraction

# -------------------------------
# 元素集合 & Grain Size 设置
# -------------------------------
elements = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
            'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
grain_sizes = [12]

# -------------------------------
# 枚举合法组合（固定 Mg=80%，Zr+Y+Gd=20%）
# -------------------------------
data = []
for zr in np.arange(0.0, 20.0 + 0.001, 0.1):  # Zr 从 0 到 20
    for y in np.arange(0.0, 20.0 - zr + 0.001, 0.2):  # Y 从 0 到 (20 - Zr)
        gd = 20.0 - zr - y
        if gd < 0:
            continue
        if (round(gd * 10) % 5) != 0:  # 例如 Gd=13.0 满足，13.3 不满足
            continue
        for grain in grain_sizes:
            comp = {el: 0.0 for el in elements}
            comp['Mg'] = 80.0
            comp['Gd'] = round(gd, 2)
            comp['Y'] = round(y, 2)
            comp['Zr'] = round(zr, 2)
            comp['Grain Size'] = grain
            data.append(comp)

# -------------------------------
# 创建 DataFrame（质量百分比）
# -------------------------------
df_wt = pd.DataFrame(data)

# -------------------------------
# 构建 Label（追踪编号，不包含 Grain Size）
# -------------------------------
def build_label(row):
    return ''.join(f"{el}-{row[el]}" for el in elements if row[el] > 0 and el != 'Grain Size')

df_wt['Label'] = df_wt.apply(build_label, axis=1)

# -------------------------------
# 转为 Composition（用于 molar fraction）
# -------------------------------
def to_comp(x):
    return Composition.from_weight_dict({el: x[el] for el in elements})

df_wt['composition'] = df_wt.apply(to_comp, axis=1)

# -------------------------------
# 计算摩尔比例（ElementFraction）
# -------------------------------
df_frac = ElementFraction().featurize_dataframe(df_wt[['composition']].copy(), col_id='composition')
df_frac = df_frac.drop(columns='composition')  # 删除中间列

# -------------------------------
# 合并：Label + Grain Size + 摩尔比例
# -------------------------------
df_final = pd.concat([df_wt[['Label', 'Grain Size']], df_frac], axis=1)

# -------------------------------
# 保存结果
# -------------------------------
output_path = '3filter_elements_composition.xlsx'
# df_final.to_excel(output_path, index=False)

print(f"✅ 共生成 {len(df_final)} 条样本（固定 Mg=80%，Zr+Y+Gd=20%），已保存为：{output_path}")


ElementFraction:   0%|          | 0/2021 [00:00<?, ?it/s]

✅ 共生成 2021 条样本（固定 Mg=80%，Zr+Y+Gd=20%），已保存为：3filter_elements_composition.xlsx


In [ ]:
df_final.to_excel(output_path, index=False)
# hall-petch筛选
import pandas as pd
from strength_calculation import calculate_Strength
elements = pd.read_excel('3filter_elements_composition.xlsx', header = 0)

df_strength = calculate_Strength(elements)
df_strength.to_excel('3filter_elements_composition_with_strength.xlsx', index = 0)

In [ ]:
# 筛选wenalloy的interant d elctrons
# 计算pymatgen特征
from pymatgen.core import Composition
# print(df)
df = pd.read_excel('3filter_elements_composition_with_strength.xlsx', header = 0)
elements = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
            'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
weightPercentage = df[elements]
# print(weightPercentage)
# 元素
# 从权重字典中提取组分信息,转换为元素序列
def to_comp(x):
    try:
        return Composition.from_weight_dict(x.to_dict())
    except Exception as ex:
        print(x, ex)

compSeries = weightPercentage.apply(to_comp, axis=1)
print(compSeries)
from matminer.featurizers.composition.composite import ElementProperty
from matminer.featurizers.composition.alloy import WenAlloys
from matminer.featurizers.composition.element import ElementFraction
from matminer.featurizers.composition.composite import Meredig
import pandas as pd
# 创建 DataFrame
from strength_calculation import calculate_Strength
df_comp = pd.DataFrame({"composition": compSeries})
from pymatgen.core import Composition

# 假设你已有 df，包含一列 composition（类型为 pymatgen 的 Composition 对象）
featurizer = WenAlloys()

# 只提取 'Interant d electrons'
featurizer.feature_labels()  # 查看所有特征名：包括 Interant d electrons
wen_df = featurizer.featurize_dataframe(df_comp.copy(), col_id='composition', ignore_errors=True)

# 查看结果（可选）
print(wen_df[['Interant d electrons']].head())
filtered_df = wen_df[wen_df['Interant d electrons'] >= 6]
print(f"筛选后样本数：{len(filtered_df)}")




0              (Mg, Gd)
1           (Mg, Gd, Y)
2           (Mg, Gd, Y)
3           (Mg, Gd, Y)
4           (Mg, Gd, Y)
             ...       
2016        (Mg, Y, Zr)
2017    (Mg, Gd, Y, Zr)
2018    (Mg, Gd, Y, Zr)
2019       (Mg, Gd, Zr)
2020           (Mg, Zr)
Length: 2021, dtype: object


WenAlloys:   0%|          | 0/2021 [00:00<?, ?it/s]

   Interant d electrons
0                   1.0
1                   2.0
2                   2.0
3                   2.0
4                   2.0
筛选后样本数：0


In [11]:
filtered_df.to_excel('3filter_wenalloy.xlsx', index = 0)

In [12]:
# 筛选wenalloy的interant d elctrons
# 计算pymatgen特征
from pymatgen.core import Composition
# print(df)
df = pd.read_excel('3filter_elements_composition_with_strength.xlsx', header = 0)
elements = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
            'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
weightPercentage = df[elements]
# print(weightPercentage)
# 元素
# 从权重字典中提取组分信息,转换为元素序列
def to_comp(x):
    try:
        return Composition.from_weight_dict(x.to_dict())
    except Exception as ex:
        print(x, ex)

compSeries = weightPercentage.apply(to_comp, axis=1)
print(compSeries)
from matminer.featurizers.composition.composite import ElementProperty
from matminer.featurizers.composition.alloy import WenAlloys
from matminer.featurizers.composition.element import ElementFraction
from matminer.featurizers.composition.composite import Meredig
import pandas as pd
# 创建 DataFrame
from strength_calculation import calculate_Strength
df_comp = pd.DataFrame({"composition": compSeries})
featurizer = ElementProperty.from_preset("magpie")
# 应用特征扩展器
df_comp = featurizer.featurize_dataframe(df_comp, "composition")
df_comp = WenAlloys().featurize_dataframe(df_comp, "composition")
df_comp = Meredig().featurize_dataframe(df_comp, "composition")
for columnName, obj in df_comp.dtypes.items():
    if obj == "object":
        print(columnName)
        del df_comp[columnName]
        
all = pd.concat([df_comp,df[['Grain Size']]],axis = 1)
# all = calculate_Strength(all)
all.to_excel('4feature_expansion.xlsx',index= 0)

0              (Mg, Gd)
1           (Mg, Gd, Y)
2           (Mg, Gd, Y)
3           (Mg, Gd, Y)
4           (Mg, Gd, Y)
             ...       
2016        (Mg, Y, Zr)
2017    (Mg, Gd, Y, Zr)
2018    (Mg, Gd, Y, Zr)
2019       (Mg, Gd, Zr)
2020           (Mg, Zr)
Length: 2021, dtype: object


ElementProperty:   0%|          | 0/2021 [00:00<?, ?it/s]

WenAlloys:   0%|          | 0/2021 [00:00<?, ?it/s]

Meredig:   0%|          | 0/2021 [00:00<?, ?it/s]

composition
Weight Fraction
Atomic Fraction


In [ ]:
from pymatgen.core import Composition
from matminer.featurizers.composition.composite import ElementProperty
from matminer.featurizers.composition.alloy import WenAlloys
from matminer.featurizers.composition.element import ElementFraction
from matminer.featurizers.composition.composite import Meredig
import pandas as pd

def to_comp(x):
    try:
        return Composition.from_weight_dict(x.to_dict())
    except Exception as ex:
        print(x, ex)

elements = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
            'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
weightPercentage = df[elements]